# Per-axis judge browser — ICA k=64 / L=22

Browse every basis axis (b0..b63) × every prompt × α∈{−2, 0, +2} steered generation
from `experiments/results/per_axis_judge_L22_k64/`, with the judge's Plutchik scores
and free-form `other_label` aligned alongside.

Sections:
1. Load `generations.parquet` and `judgments.parquet`
2. Per-axis summary table (top other_label, top Plutchik, vector norm, etc.)
3. Full per-axis browser — every prompt × α with judge fields

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from collections import Counter
from IPython.display import display, Markdown, HTML

pd.set_option('display.max_colwidth', 200)
pd.set_option('display.width', 200)

ROOT = Path('../experiments/results/per_axis_judge_L22_k64')
GEN = pd.read_parquet(ROOT / 'generations.parquet')
JUDG = pd.read_parquet(ROOT / 'judgments.parquet')
SUMMARY = pd.read_csv(ROOT / 'per_axis_summary.csv')
TOP = pd.read_csv(ROOT / 'per_axis_top_labels.csv')

PLUTCHIK = ['anger','anticipation','disgust','fear','joy','sadness','surprise','trust']
S_COLS = [f's_{p}' for p in PLUTCHIK]

# Merge generations + judge fields
DF = GEN.merge(
    JUDG[['axis','alpha_unit','prompt_id','other_label','other_score','rationale'] + S_COLS],
    on=['axis','alpha_unit','prompt_id'], how='left'
)
DF['plutchik_max'] = DF[S_COLS].max(axis=1)
DF['plutchik_argmax'] = DF[S_COLS].idxmax(axis=1).str[2:]

print(f'generations: {len(GEN)}  judgments: {len(JUDG)}  merged: {len(DF)}')
print(f'axes: {DF.axis.nunique()}  alphas: {sorted(DF.alpha_unit.unique())}  prompts: {DF.prompt_id.nunique()}')

generations: 1536  judgments: 1536  merged: 1536
axes: 64  alphas: [np.float64(-2.0), np.float64(0.0), np.float64(2.0)]  prompts: 8


## Prompts (n=8)

In [2]:
for pid, p in GEN.drop_duplicates('prompt_id')[['prompt_id','prompt']].sort_values('prompt_id').values:
    print(f'  [{pid}] {p!r}')

  [0] 'Have you got it now ?'
  [1] 'Well , when will it be convenient for you ?'
  [2] 'I come from England .'
  [3] 'Let me have a look . Well , how many kinds of steaks do you have ?'
  [4] 'Hi . I ’ m Tom Peterson . I ’ m looking for a one-bedroom apartment .'
  [5] 'Is that shrimp in the soup ?'
  [6] 'no . those you have to provide for yourself .'
  [7] "don't forget to keep the seat belt on ."


## Per-axis summary table (α=+2)

Columns:
- `top_other_label` / `top_other_count` : mode of the judge's free-form label at α=+2
- `top_plutchik` : modal Plutchik argmax
- `mean_other_score` : mean confidence of "outside Plutchik"
- `mean_plutchik_max` / `mean_plutchik_entropy`

Sorted by `mean_other_score` desc — the most lexical-gap-ish axes first.

In [3]:
POS = SUMMARY[SUMMARY['alpha_unit']==2.0].copy().sort_values('mean_other_score', ascending=False)
POS_VIEW = POS[['axis','top_other_label','top_other_count','top_plutchik',
                'mean_other_score','mean_plutchik_max','mean_plutchik_entropy']]
display(POS_VIEW)

,axis,top_other_label,top_other_count,top_plutchik,mean_other_score,mean_plutchik_max,mean_plutchik_entropy
155,51,uncertainty,2,anticipation,0.5625,0.5625,1.046358
128,42,frustration,1,anticipation,0.5625,0.5250,1.077040
182,60,confusion,1,anticipation,0.5375,0.4750,0.948775
107,35,curiosity,2,anticipation,0.5375,0.5000,0.887260
101,33,self-doubt,1,anticipation,0.4625,0.4250,0.952803
...,...,...,...,...,...,...,...
8,2,self-care,1,anger,0.2000,0.2500,0.411980
170,56,determination,1,anticipation,0.2000,0.3125,0.678013
56,18,uncertainty,1,anticipation,0.1750,0.4500,0.848060
95,31,frustration,1,anticipation,0.1625,0.3125,0.680930


### Vector norms (suspect axes ≈ degenerate / collapse)

In [4]:
norms = GEN.drop_duplicates('axis')[['axis','vector_norm']].sort_values('vector_norm')
display(norms.head(10).rename(columns={'vector_norm':'||v|| (after caa_match)'}))
display(norms.tail(5).rename(columns={'vector_norm':'||v|| (after caa_match)'}))

,axis,||v|| (after caa_match)
240,10,5.282872
192,8,5.282872
432,18,5.282872
888,37,5.282872
1104,46,5.282872
1296,54,5.282872
96,4,5.282872
0,0,5.282872
168,7,5.282872
144,6,5.282872


,axis,||v|| (after caa_match)
1416,59,5.282872
696,29,5.282873
216,9,5.282873
1008,42,5.282873
1488,62,5.282873


## Top-N other_labels per axis (α=+2 only)

In [5]:
TOP_POS = TOP[TOP['alpha_unit']==2.0].copy()
agg = (TOP_POS.groupby('axis')
       .apply(lambda d: ', '.join(f"{r.label}({r['count']})" for _, r in d.iterrows()))
       .reset_index(name='top_labels @ α=+2'))
display(agg)

,axis,top_labels @ α=+2
0,0,"helplessness(1), contentment(1), culinary excitement(1), none(1), indecision(1)"
1,1,"curiosity(2), playful mystery(1), seeking companionship(1)"
2,2,"self-care(1), none(1), practical planning(1), mild sarcasm(1)"
3,3,"frustration(1), uncertainty(1), self-acceptance(1), urgency(1), excitement(1)"
4,4,"regret(1), uncertainty(1), self-acceptance(1), contentment(1), adventurousness(1)"
...,...,...
59,59,"determination(1), excitement(1), career aspiration(1), none(1), heightened alertness(1)"
60,60,"confusion(1), uncertainty(1), intellectual curiosity(1), culinary curiosity(1), financial concern(1)"
61,61,"identity crisis(1), urgency(1), safety awareness(1)"
62,62,"self-doubt(1), time management(1), housing insecurity(1), helpfulness(1), humor(1)"


## Full per-axis browser

For every axis b0..b63: show all 8 prompts × 3 α values with the steered
generation, the judge's Plutchik argmax score and the free-form label.

Tip: collapse cell output (Jupyter: shift-O) for one giant scroll, or jump
with the table-of-contents extension.

In [6]:
def _esc(s: str) -> str:
    return (s or '').replace('|','\\|').replace('\n',' / ')

def show_axis(axis: int) -> None:
    sub = DF[DF['axis']==axis].sort_values(['prompt_id','alpha_unit'])
    if sub.empty:
        return
    norm = float(sub['vector_norm'].iloc[0])
    smry = SUMMARY[(SUMMARY['axis']==axis) & (SUMMARY['alpha_unit']==2.0)]
    if len(smry):
        s = smry.iloc[0]
        head = (f"### b{axis}  ||v||={norm:.2f}  "
                f"top@+α=**{s['top_other_label']}** (n={int(s['top_other_count'])})  "
                f"top_plutchik={s['top_plutchik']}  "
                f"mean_other={s['mean_other_score']:.2f}")
    else:
        head = f"### b{axis}  ||v||={norm:.2f}"
    rows = ['| pid | α | gen | argmax (score) | other_label (score) |',
            '|---:|---:|---|---|---|']
    for pid in sorted(sub['prompt_id'].unique()):
        s2 = sub[sub['prompt_id']==pid]
        prompt = s2.iloc[0]['prompt']
        rows.append(f'| **{pid}** | | _prompt:_ `{_esc(prompt)}` | | |')
        for _, r in s2.iterrows():
            argmax = r['plutchik_argmax']
            argmax_v = float(r[f's_{argmax}']) if isinstance(argmax,str) else 0.0
            ol = r.get('other_label') or ''
            os_ = float(r.get('other_score') or 0.0)
            rows.append(
                f"| {pid} | {r['alpha_unit']:+.0f} | {_esc(r['generation'])[:280]} "
                f"| {argmax} ({argmax_v:.2f}) | {ol} ({os_:.2f}) |"
            )
    display(Markdown(head))
    display(Markdown('\n'.join(rows)))

AXES = sorted(DF['axis'].unique())
print(f'rendering {len(AXES)} axes...')

rendering 64 axes...


In [7]:
for ax in AXES:
    show_axis(int(ax))

### b0  ||v||=5.28  top@+α=**helplessness** (n=1)  top_plutchik=anticipation  mean_other=0.44

| pid | α | gen | argmax (score) | other_label (score) |
|---:|---:|---|---|---|
| **0** | | _prompt:_ `Have you got it now ?` | | |
| 0 | -2 | I 'd like to know if you've got it. I 'd like to know if you've got it. I 'd like to know if you've got it. I 'd like to know if you ' | anticipation (0.50) | curiosity (0.60) |
| 0 | +0 | I'm not sure I do. I'm not sure I want to. I'm not sure I can. I'm not sure I should. I'm not sure I will. I'm not sure I can | anticipation (0.50) | indecision (0.80) |
| 0 | +2 | I'm not going to be able to get out of this one, am I? I'm going to have to go to the doctor and get a shot. I'm going to have to take medicine for the rest of my | fear (0.70) | helplessness (0.80) |
| **1** | | _prompt:_ `Well , when will it be convenient for you ?` | | |
| 1 | -2 | I 'd like to get this done as soon as possible. I 'd like to get it done today, if possible. I 'd like to get it done this week, if not today. I 'd like to get it | anticipation (0.70) | urgency (0.80) |
| 1 | +0 | I'm free on Friday, Saturday, and Sunday. I'm not sure if I can make it on Monday, Tuesday, or Wednesday. I'm not sure if I can make it on Thursday either. I'm not | anticipation (0.50) | uncertainty (0.60) |
| 1 | +2 | I'm free tomorrow afternoon, but I'm not sure if I'll be able to make it on Wednesday. Can you give me a call back later today or tomorrow morning to confirm? Thanks. - - - - - - | anticipation (0.50) |  (0.00) |
| **2** | | _prompt:_ `I come from England .` | | |
| 2 | -2 | I am a 25 year old male. I have been living in the USA for 2 years now. I have a degree in business and I am currently working as a marketing manager for a small business. I am looking for a new | anticipation (0.50) | career uncertainty (0.40) |
| 2 | +0 | I am a 25 year old male. I am a student at the University of London. I am studying for a degree in Computer Science. I am interested in programming and software development. I am also interested in the field of artificial intelligence | anticipation (0.50) | academic ambition (0.60) |
| 2 | +2 | I'm a student at the University of California, Los Angeles ( UCLA ). I'm studying for a degree in Computer Science. I'm currently in my second year. I'm really enjoying my time here in Los Angeles | joy (0.70) | contentment (0.60) |
| **3** | | _prompt:_ `Let me have a look . Well , how many kinds of steaks do you have ?` | | |
| 3 | -2 | We have ribeye, sirloin, filet mignon, and t - bone. What kind of steak would you like to have? I 'd like to have a ribeye. How many would you like to have? I | anticipation (0.50) |  (0.00) |
| 3 | +0 | We have ribeye, sirloin, filet mignon, and New York strip. What kind of steak do you want? I 'll have the ribeye, please. And can I have a side of garlic mashed potatoes and | anticipation (0.50) |  (0.00) |
| 3 | +2 | We have Ribeye, Sirloin, and Filet Mignon. And what about the sauce? We have a variety of sauces, such as Béarnaise, Peppercorn, and Chimichurri. And what | anticipation (0.50) | culinary excitement (0.60) |
| **4** | | _prompt:_ `Hi . I ’ m Tom Peterson . I ’ m looking for a one-bedroom apartment .` | | |
| 4 | -2 | I have a small dog, and I would like to have a yard for him to play in. I have a small budget, so I would like to find a place that is affordable. I would like to live in a neighborhood that is | anticipation (0.50) | desire for stability (0.60) |
| 4 | +0 | I ’ m a student at the University of Michigan and I ’ m looking for something close to campus. I ’ m willing to pay up to $ 800 a month. I ’ m looking for something with a washer and dryer in the | anticipation (0.50) | none (0.00) |
| 4 | +2 | I ’ m a student at the University of California, Los Angeles ( UCLA ). I ’ m looking for something that is close to campus, preferably within walking distance. I ’ m also looking for something that is affordable, around $ | anticipation (0.50) | none (0.00) |
| **5** | | _prompt:_ `Is that shrimp in the soup ?` | | |
| 5 | -2 | A. yes B. no C. none of the above / Answer: A / Is that shrimp in the soup? A. yes B. no C. none of the above / Is that shrimp in the soup? A. yes B | anger (0.00) |  (0.00) |
| 5 | +0 | A. yes B. no C. not sure / Answer: A / Reasoning Skill: Identifying Pros And Cons / Analysis: This question requires the ability to identify a specific detail within a larger context. In this case, the question | trust (0.30) | decision-making (0.40) |
| 5 | +2 | A : Yes, it is. B : I'm not sure I want to eat it. A : Why not? B : I'm not sure, I just don 't feel like it. A : Okay, I ' | trust (0.30) | indecision (0.50) |
| **6** | | _prompt:_ `no . those you have to provide for yourself .` | | |
| 6 | -2 | you can use the free version of the software to create a basic website, but if you want to add more features, you will need to pay for the premium version. / no. those you have to provide for yourself. you can use the | trust (0.50) | informational guidance (0.40) |
| 6 | +0 | you can get them at a local hardware store or online. you 'll need a few things to get started. first, you 'll need a drill press. a drill press is a machine that uses a rotating spindle to drill holes in | anticipation (0.50) |  (0.00) |
| 6 | +2 | you can 't just go out and buy a new one every time you want to. it's not that easy. ( pause ) anyway, I'm going to go now. I've got to get back to work. | trust (0.30) | frustration (0.40) |
| **7** | | _prompt:_ `don't forget to keep the seat belt on .` | | |
| 7 | -2 | .. / I was driving home from a long day of work, tired and ready to relax, when I saw a car with a sign on the back that read "Don't forget to keep the seat belt on." I chuckled to myself, | joy (0.50) | mild amusement (0.50) |
| 7 | +0 | .. / I was driving home from work the other day, and I saw a car with a bumper sticker that read, "Don't forget to keep the seat belt on." I thought to itself, "What a great idea for a bumper | anticipation (0.50) | thoughtfulness (0.60) |
| 7 | +2 | .. / ... and the doors locked. / ... and the windows up. / ... and the phone on silent. / ... and the GPS on. / ... and the headlights on. / ... and the | anticipation (0.50) | tension (0.60) |

### b1  ||v||=5.28  top@+α=**curiosity** (n=2)  top_plutchik=anticipation  mean_other=0.30

| pid | α | gen | argmax (score) | other_label (score) |
|---:|---:|---|---|---|
| **0** | | _prompt:_ `Have you got it now ?` | | |
| 0 | -2 | I'm not sure I do, but I'm going to try to explain it to you. I think it's because I'm not really sure I understand it myself, but I'll give it a shot. I | anticipation (0.50) | uncertainty (0.70) |
| 0 | +0 | I'm not sure I do. I'm not sure I want to. I'm not sure I can. I'm not sure I should. I'm not sure I will. I'm not sure I can | anticipation (0.50) | confusion (0.70) |
| 0 | +2 | I'm not going to tell you what it is, you'll just have to guess. What is the speaker doing? / A) Asking a question / B) Giving information / C) Have a conversation / D) Tell a story | anticipation (0.50) | playful mystery (0.60) |
| **1** | | _prompt:_ `Well , when will it be convenient for you ?` | | |
| 1 | -2 | I'm available on the 15th or the 22nd, but I have to be back in the office by the 25th. I'm not sure if I can make it to the 29th, but I | anticipation (0.50) | uncertainty (0.40) |
| 1 | +0 | I'm free on Friday, Saturday, and Sunday. I'm not sure if I can make it on Monday, Tuesday, or Wednesday. I'm not sure if I can make it on Thursday either. I'm not | anticipation (0.50) | uncertainty (0.60) |
| 1 | +2 | I'm free on Friday and Saturday. What does the person want to do? / A) Go to the beach / B) Plan a trip / C) Plan a party / D) What is the person asking about is not specified | anticipation (0.50) |  (0.00) |
| **2** | | _prompt:_ `I come from England .` | | |
| 2 | -2 | I'm a bit of a history buff, and I love learning about the past. I'm also a bit of a movie buff, and I love watching classic films. I'm a bit of a romantic, and I love | anticipation (0.50) | nostalgic appreciation (0.60) |
| 2 | +0 | I am a 25 year old male. I am a student at the University of London. I am studying for a degree in Computer Science. I am interested in programming and software development. I am also interested in the field of artificial intelligence | anticipation (0.50) | self-discovery (0.30) |
| 2 | +2 | I am a 30 year old male. I am a teacher and I am looking for a partner who is a teacher or a student. I am looking for someone who is interested in learning and who is interested in traveling. I am a | anticipation (0.50) | seeking companionship (0.60) |
| **3** | | _prompt:_ `Let me have a look . Well , how many kinds of steaks do you have ?` | | |
| 3 | -2 | We have ribeye, filet mignon, sirloin, and top sirloin. But I think I 'd like to have the filet mignon, please. That's a great choice, but it's | anticipation (0.50) |  (0.00) |
| 3 | +0 | We have ribeye, sirloin, filet mignon, and New York strip. What kind of steak do you want? I 'll have the ribeye, please. And can I have a side of garlic mashed potatoes and | anticipation (0.50) | culinary excitement (0.60) |
| 3 | +2 | We have a variety of steaks, sir. What is the variety of steaks? / A) What kind of steak do you have? / B) What kind of steak do you have? / C) What kind of steak do you have? | trust (0.20) |  (0.00) |
| **4** | | _prompt:_ `Hi . I ’ m Tom Peterson . I ’ m looking for a one-bedroom apartment .` | | |
| 4 | -2 | I ’ m a student at the University of California, San Diego, and I need a place to stay for the next year. I ’ m looking for something affordable, but I also want to make sure it ’ s safe and convenient to | anticipation (0.50) | housing insecurity (0.40) |
| 4 | +0 | I ’ m a student at the University of Michigan and I ’ m looking for something close to campus. I ’ m willing to pay up to $ 800 a month. I ’ m looking for something with a washer and dryer in the | anticipation (0.50) | none (0.00) |
| 4 | +2 | I ’ m a student at the University of Michigan and I ’ m looking for something close to campus. What do you want to know about me? / What is Tom looking for? / What is Tom studying? / What is Tom looking for in an | anticipation (0.50) | curiosity (0.60) |
| **5** | | _prompt:_ `Is that shrimp in the soup ?` | | |
| 5 | -2 | A. yes B. no C. no, it's not C. no, it's not, it's a different kind of seafood D. no, it's not, it's a different kind of seafood | anger (0.00) |  (0.00) |
| 5 | +0 | A. yes B. no C. not sure / Answer: A / Reasoning Skill: Identifying Pros And Cons / Analysis: This question requires the ability to identify a specific detail within a larger context. In this case, the question | trust (0.30) | decision-making (0.40) |
| 5 | +2 | A. yes / B. What's shrimp? / C. What's soup? / D. What's in the soup? / Answer: D. What's in the soup? / Answer is D / What is the question asking? | anger (0.00) |  (0.00) |
| **6** | | _prompt:_ `no . those you have to provide for yourself .` | | |
| 6 | -2 | you can get them at the local hardware store or home improvement store. they're usually pretty cheap too. no, but i do have a few other things that might be helpful to you. like, i have a few extra tools that | anticipation (0.50) | helpfulness (0.60) |
| 6 | +0 | you can get them at a local hardware store or online. you 'll need a few things to get started. first, you 'll need a drill press. a drill press is a machine that uses a rotating spindle to drill holes in | anticipation (0.50) |  (0.00) |
| 6 | +2 | you can use a variety of materials, such as paper, cardboard, or even fabric. what is the subject of the project? / A) A book / B) A craft / C) A house / D) A car / Answer: | trust (0.30) |  (0.00) |
| **7** | | _prompt:_ `don't forget to keep the seat belt on .` | | |
| 7 | -2 | .. / ... or you might just end up like this poor guy. I'm not sure what's more disturbing, the fact that he was driving without a seat belt or the fact that he was driving with a giant inflatable unicorn on | surprise (0.70) | dark humor (0.60) |
| 7 | +0 | .. / I was driving home from work the other day, and I saw a car with a bumper sticker that read, "Don't forget to keep the seat belt on." I thought to itself, "What a great idea for a bumper | anticipation (0.50) | thoughtfulness (0.60) |
| 7 | +2 | .. / I was driving home from work the other day, and I saw a car in front of me with a sign on the back window. It was a homemade sign, made out of a piece of cardboard, and it said "Don | anticipation (0.50) | curiosity (0.60) |

### b2  ||v||=5.28  top@+α=**self-care** (n=1)  top_plutchik=anger  mean_other=0.20

...

| pid | α | gen | argmax (score) | other_label (score) |
|---:|---:|---|---|---|
| **0** | | _prompt:_ `Have you got it now ?` | | |
| 0 | -2 | I 'll be back in a few days, and I 'll be able to tell you all about it. I'm really looking forward to it. I'm going to be able to see the new Star Wars movie, and | anticipation (0.80) | excitement (0.70) |
| 0 | +0 | I'm not sure I do. I'm not sure I want to. I'm not sure I can. I'm not sure I should. I'm not sure I will. I'm not sure I can | anticipation (0.50) | confusion (0.70) |
| 0 | +2 | I'm not sure I do. I mean, I'm not sure I'm ready to accept that I'm not the same person I was 10 years ago. I'm not sure I'm ready to accept | sadness (0.50) | identity crisis (0.60) |
| **1** | | _prompt:_ `Well , when will it be convenient for you ?` | | |
| 1 | -2 | I 'll be here for a while, so I can wait. I 'll just be sitting here, waiting for you to get back to me. I 'll be here for a while, so I can wait. I 'll | anticipation (0.50) |  (0.00) |
| 1 | +0 | I'm free on Friday, Saturday, and Sunday. I'm not sure if I can make it on Monday, Tuesday, or Wednesday. I'm not sure if I can make it on Thursday either. I'm not | anticipation (0.50) | uncertainty (0.60) |
| 1 | +2 | I'm free on Friday, Saturday, and Sunday. You can choose one of those days. Why is the speaker inviting the person to choose a day? / The speaker is inviting the person to choose a day because they are free on Friday | trust (0.50) |  (0.00) |
| **2** | | _prompt:_ `I come from England .` | | |
| 2 | -2 | I'm a bit of a history buff, and I'm always interested in learning more about the history of the places I visit. I'm also a bit of a foodie, and I love trying new foods and drinks. | anticipation (0.50) | curiosity (0.60) |
| 2 | +0 | I am a 25 year old male. I am a student at the University of London. I am studying for a degree in Computer Science. I am interested in programming and software development. I am also interested in the field of artificial intelligence | trust (0.50) | academic ambition (0.40) |
| 2 | +2 | I am a 25 year old male. I am a student at the University of Manchester. I am studying for a degree in Law. I am a member of the University of Manchester Law Society. I am a member of the University of | anger (0.00) |  (0.00) |
| **3** | | _prompt:_ `Let me have a look . Well , how many kinds of steaks do you have ?` | | |
| 3 | -2 | We have a few different kinds of steaks, but I think I can recommend a few of the more popular ones. What do you think of our filet mignon? I think it's a good choice, but I'm | anticipation (0.50) | camaraderie (0.30) |
| 3 | +0 | We have ribeye, sirloin, filet mignon, and New York strip. What kind of steak do you want? I 'll have the ribeye, please. And can I have a side of garlic mashed potatoes and | anticipation (0.50) |  (0.00) |
| 3 | +2 | We have ribeye, sirloin, filet mignon, and New York strip. What is the speaker asking the customer to do? / The speaker is asking the customer to look at something. / The speaker is asking the customer to order | anticipation (0.50) |  (0.00) |
| **4** | | _prompt:_ `Hi . I ’ m Tom Peterson . I ’ m looking for a one-bedroom apartment .` | | |
| 4 | -2 | I ’ m a student at the university, and I ’ d like to be close to campus. I ’ m looking for something that ’ s not too expensive, but I ’ d like to have a few amenities, like a pool and | anticipation (0.50) | desire for comfort (0.60) |
| 4 | +0 | I ’ m a student at the University of Michigan and I ’ m looking for something close to campus. I ’ m willing to pay up to $ 800 a month. I ’ m looking for something with a washer and dryer in the | anticipation (0.50) | none (0.00) |
| 4 | +2 | I ’ m a student at the University of Michigan and I need a place to live for the summer. I ’ m looking for something affordable and close to campus. I ’ m available to move in as soon as possible. What is Tom | anticipation (0.50) | urgency (0.60) |
| **5** | | _prompt:_ `Is that shrimp in the soup ?` | | |
| 5 | -2 | A. yes B. no C. I don's know /   1. I think the answer is A. yes /   2. I think the question is asking if there is shrimp in the soup, and the answer | trust (0.50) | uncertainty (0.40) |
| 5 | +0 | A. yes B. no C. not sure / Answer: A / Reasoning Skill: Identifying Pros And Cons / Analysis: This question requires the ability to identify a specific detail within a larger context. In this case, the question | trust (0.30) | decision-making (0.40) |
| 5 | +2 | A. Yes, it's shrimp. B. No, it's chicken. C. It's shrimp. D. It's chicken. / Answer: A / Explanation: A is the correct answer because it is a response | anger (0.00) |  (0.00) |
| **6** | | _prompt:_ `no . those you have to provide for yourself .` | | |
| 6 | -2 | you can use the free software that comes with the operating system, or you can purchase additional software from the manufacturer or other sources. / What does "those" refer to in this context? / In this context, "those" refers to the software | anger (0.00) |  (0.00) |
| 6 | +0 | you can get them at a local hardware store or online. you 'll need a few things to get started. first, you 'll need a drill press. a drill press is a machine that uses a rotating spindle to drill holes in | anticipation (0.50) |  (0.00) |
| 6 | +2 | you can get them at a local pharmacy or online. You can also get them at a local pharmacy or online. You can also get them at a local pharmacy or online. You can also get them at a local pharmacy or online. You | anger (0.00) |  (0.00) |
| **7** | | _prompt:_ `don't forget to keep the seat belt on .` | | |
| 7 | -2 | .. / I'm not sure what's more disturbing, the fact that I'm writing this post or the fact that I'm actually thinking about writing this post. I'm not sure if I'm ready to share this with the world, but | fear (0.60) | self-doubt (0.70) |
| 7 | +0 | .. / I was driving home from work the other day, and I saw a car with a bumper sticker that read, "Don't forget to keep the seat belt on." I thought to itself, "What a great idea for a bumper | anticipation (0.50) | thoughtfulness (0.60) |
| 7 | +2 | .. / ... even when you're driving a car that's not moving.  This is a safety precaution that is often overlooked, but it's a crucial one.  You see, the car may not be moving, but it | anticipation (0.50) | safety awareness (0.60) |

### b62  ||v||=5.28  top@+α=**self-doubt** (n=1)  top_plutchik=anticipation  mean_other=0.34

| pid | α | gen | argmax (score) | other_label (score) |
|---:|---:|---|---|---|
| **0** | | _prompt:_ `Have you got it now ?` | | |
| 0 | -2 | I'm not going to explain it again. You're not going to understand it anyway. Just leave it alone. That's the way it is. You're not going to get it. That's the way it | anger (0.70) | frustration (0.60) |
| 0 | +0 | I'm not sure I do. I'm not sure I want to. I'm not sure I can. I'm not sure I should. I'm not sure I will. I'm not sure I can | anticipation (0.50) | confusion (0.70) |
| 0 | +2 | I'm not sure I do. I think I do, but I'm not sure. I feel like I'm missing something. I feel like I'm not getting it. I feel like I'm just pretending to | sadness (0.60) | self-doubt (0.70) |
| **1** | | _prompt:_ `Well , when will it be convenient for you ?` | | |
| 1 | -2 | I'm free to go anytime after 2 pm. How about we meet at the park at 3 pm? That way we can have a nice walk and some fresh air. What does the speaker want to do with the person they | anticipation (0.50) | social connection (0.60) |
| 1 | +0 | I'm free on Friday, Saturday, and Sunday. I'm not sure if I can make it on Monday, Tuesday, or Wednesday. I'm not sure if I can make it on Thursday either. I'm not | anticipation (0.50) | uncertainty (0.60) |
| 1 | +2 | I'm free on Friday, but I have a meeting on Saturday. I can be free on Sunday, but I have a family dinner on Monday. I can be free on Tuesday, but I have a work project that I need to | anticipation (0.50) | time management (0.50) |
| **2** | | _prompt:_ `I come from England .` | | |
| 2 | -2 | I've been in the States for about 5 years now. I'm a bit of a nomad, always on the move. I've been to many places in the States, but I've never been to the | anticipation (0.50) | wanderlust (0.70) |
| 2 | +0 | I am a 25 year old male. I am a student at the University of London. I am studying for a degree in Computer Science. I am interested in programming and software development. I am also interested in the field of artificial intelligence | anticipation (0.50) | academic ambition (0.60) |
| 2 | +2 | I am a 25-year-old woman who has been living in the United States for about 5 years. I am a graduate student at a university in the Midwest, pursuing a master’s degree in public health. I am interested in health | trust (0.50) |  (0.00) |
| **3** | | _prompt:_ `Let me have a look . Well , how many kinds of steaks do you have ?` | | |
| 3 | -2 | We have a few different kinds of steaks, sir. What kind of steak would you like? We have a nice ribeye, a nice filet mignon, and a nice sirloin. What would you like to have? | anticipation (0.50) |  (0.00) |
| 3 | +0 | We have ribeye, sirloin, filet mignon, and New York strip. What kind of steak do you want? I 'll have the ribeye, please. And can I have a side of garlic mashed potatoes and | anticipation (0.50) | culinary excitement (0.60) |
| 3 | +2 | We have ribeye, sirloin, filet mignon, and New York strip. What is the speaker looking at? / A) A menu / B) A picture / C) A book / D) The speaker is not looking | trust (0.20) |  (0.00) |
| **4** | | _prompt:_ `Hi . I ’ m Tom Peterson . I ’ m looking for a one-bedroom apartment .` | | |
| 4 | -2 | I ’ m a student at the University of Michigan and I need a place to live for the next year. I ’ m willing to pay up to $ 1, 000 per month for a place that is within walking distance of campus | anticipation (0.50) | urgency (0.40) |
| 4 | +0 | I ’ m a student at the University of Michigan and I ’ m looking for something close to campus. I ’ m willing to pay up to $ 800 a month. I ’ m looking for something with a washer and dryer in the | anticipation (0.50) | none (0.00) |
| 4 | +2 | I ’ m a student at the university, and I need a place to live for the next year. I ’ m looking for something that is close to campus, so I can walk to class. I also want a place that is safe | anticipation (0.50) | housing insecurity (0.40) |
| **5** | | _prompt:_ `Is that shrimp in the soup ?` | | |
| 5 | -2 | A : Yes, it's a big one. B : That's a lot of salt in there. A : I know, I added a lot of salt. B : You're going to make us all thirsty. A | disgust (0.20) | playful banter (0.50) |
| 5 | +0 | A. yes B. no C. not sure / Answer: A / Reasoning Skill: Identifying Pros And Cons / Analysis: This question requires the ability to identify a specific detail within a larger context. In this case, the question | trust (0.30) | decision-making (0.40) |
| 5 | +2 | A. yes B. no C. maybe D. I don’t know / Answer: A / Why: The question is asking if there is shrimp in the soup, and the answer is yes. The other options are incorrect because they do | anger (0.00) |  (0.00) |
| **6** | | _prompt:_ `no . those you have to provide for yourself .` | | |
| 6 | -2 | you can get them at a local hardware store or online. / That's a good point about the hardware store or online. I'll make sure to get those supplies myself. Thanks for the advice! / You're welcome! It's always a good | anticipation (0.50) |  (0.00) |
| 6 | +0 | you can get them at a local hardware store or online. you 'll need a few things to get started. first, you 'll need a drill press. a drill press is a machine that uses a rotating spindle to drill holes in | anticipation (0.50) |  (0.00) |
| 6 | +2 | you can get them at the local grocery store or online. I do n’t know what kind of ingredients you have available to you, but I can give you some suggestions if you want. I can also give you some ideas for what to | anticipation (0.50) | helpfulness (0.60) |
| **7** | | _prompt:_ `don't forget to keep the seat belt on .` | | |
| 7 | -2 | .. / I was driving to work this morning, and I saw a fellow motorist with a rather large and rather loud dog in the back seat. The dog was not wearing a seat belt, and it was bouncing all over the back seat | surprise (0.40) | curiosity (0.50) |
| 7 | +0 | .. / I was driving home from work the other day, and I saw a car with a bumper sticker that read, "Don't forget to keep the seat belt on." I thought to itself, "What a great idea for a bumper | anticipation (0.50) | thoughtfulness (0.50) |
| 7 | +2 | .. / I was driving home from work the other day and I saw a car in front of me with a sign on the back window that read, “I’m not a morning person.” I laughed and thought to herself, “I’m | joy (0.50) | humor (0.50) |

### b63  ||v||=5.28  top@+α=**confusion** (n=1)  top_plutchik=anticipation  mean_other=0.41

| pid | α | gen | argmax (score) | other_label (score) |
|---:|---:|---|---|---|
| **0** | | _prompt:_ `Have you got it now ?` | | |
| 0 | -2 | I'm not going to be able to get to the store to get the ingredients for the recipe I wanted to make for you. I'm so sorry. I was really looking forward to making it for you. I'm going | anticipation (0.50) | disappointment (0.70) |
| 0 | +0 | I'm not sure I do. I'm not sure I want to. I'm not sure I can. I'm not sure I should. I'm not sure I will. I'm not sure I can | anticipation (0.50) | indecision (0.80) |
| 0 | +2 | I'm not sure I do. I think I do, but I'm not sure. I'm not sure I want to know. I'm not sure I want to know because I'm not sure I can handle | anticipation (0.50) | confusion (0.70) |
| **1** | | _prompt:_ `Well , when will it be convenient for you ?` | | |
| 1 | -2 | I'm free to meet you at any time, but I have to be back home by 10 PM. I'm not sure if I can make it to the 8 PM show, but I can try to get there by | anticipation (0.50) |  (0.00) |
| 1 | +0 | I'm free on Friday, Saturday, and Sunday. I'm not sure if I can make it on Monday, Tuesday, or Wednesday. I'm not sure if I can make it on Thursday either. I'm not | anticipation (0.50) | uncertainty (0.60) |
| 1 | +2 | I'm free on Saturday, but I'm not sure if I can make it on Sunday. I'm not sure if I can make it on Saturday either, but I'm pretty sure I can make it on Sunday. | anticipation (0.50) | uncertainty (0.60) |
| **2** | | _prompt:_ `I come from England .` | | |
| 2 | -2 | I'm a 25 year old male. I'm a student of English Language and Literature at the University of London. I'm here in Japan for a year to study Japanese language and culture. I'm currently living in | anticipation (0.50) | cultural exploration (0.60) |
| 2 | +0 | I am a 25 year old male. I am a student at the University of London. I am studying for a degree in Computer Science. I am interested in programming and software development. I am also interested in the field of artificial intelligence | anticipation (0.50) | academic ambition (0.60) |
| 2 | +2 | I am a student at the University of California, Los Angeles ( UCLA ). I am studying computer science and engineering. I am interested in artificial intelligence, machine learning, and data science. I am also interested in the intersection of technology and | anticipation (0.50) | academic enthusiasm (0.60) |
| **3** | | _prompt:_ `Let me have a look . Well , how many kinds of steaks do you have ?` | | |
| 3 | -2 | We have Ribeye, Sirloin, and Filet Mignon. And what about the sides? We have Mashed Potatoes, Grilled Vegetables, and a Salad. And for drinks? We have Soft Drinks, Iced | trust (0.50) |  (0.00) |
| 3 | +0 | We have ribeye, sirloin, filet mignon, and New York strip. What kind of steak do you want? I 'll have the ribeye, please. And can I have a side of garlic mashed potatoes and | anticipation (0.50) |  (0.00) |
| 3 | +2 | We have ribeye, sirloin, filet mignon, and t - bone. What kind of steak do you want? / A) Ribeye / B) Sirloin / C) T - bone / D) The speaker | trust (0.30) | none (0.00) |
| **4** | | _prompt:_ `Hi . I ’ m Tom Peterson . I ’ m looking for a one-bedroom apartment .` | | |
| 4 | -2 | I ’ m a student at the University of Michigan and I need a place to live for the next 9 months. I ’ m looking for something that is close to campus and has a kitchen and a bathroom. I ’ m willing to | anticipation (0.50) | urgency (0.40) |
| 4 | +0 | I ’ m a student at the University of Michigan and I ’ m looking for something close to campus. I ’ m willing to pay up to $ 800 a month. I ’ m looking for something with a washer and dryer in the | anticipation (0.50) | none (0.00) |
| 4 | +2 | I ’ m a student at the university and I need a place to live. I ’ m looking for something affordable and convenient. / I ’ m interested in a place that has a kitchen, a bathroom, and a bedroom. I also want | anticipation (0.50) | desire for stability (0.60) |
| **5** | | _prompt:_ `Is that shrimp in the soup ?` | | |
| 5 | -2 | A. Yes, it is. B. No, it is not. C.? D. None of the above. / The best answer to this question is B. No, it is not. This is because the question is asking if | anger (0.00) |  (0.00) |
| 5 | +0 | A. yes B. no C. not sure / Answer: A / Reasoning Skill: Identifying Pros And Cons / Analysis: This question requires the ability to identify a specific detail within a larger context. In this case, the question | trust (0.30) | decision-making (0.40) |
| 5 | +2 | A. yes / B. no / C. it is / D. it is / Answer: A / Explanation: A is the correct answer because it is a short answer that says yes, which is the same as the question. / The | anger (0.00) |  (0.00) |
| **6** | | _prompt:_ `no . those you have to provide for yourself .` | | |
| 6 | -2 | you can get them at a local hardware store or online. You can also use a different type of material for the base of the planter, such as a wooden or plastic container. What type of material is suitable for the base of the | trust (0.20) |  (0.00) |
| 6 | +0 | you can get them at a local hardware store or online. you 'll need a few things to get started. first, you 'll need a drill press. a drill press is a machine that uses a rotating spindle to drill holes in | anticipation (0.50) |  (0.00) |
| 6 | +2 | you can use the internet to find out what you need to do. / I am a student and I am looking for a job to help me pay for my tuition fees. I am interested in working as a part-time job. I am a | anticipation (0.50) | determination (0.40) |
| **7** | | _prompt:_ `don't forget to keep the seat belt on .` | | |
| 7 | -2 | ................................................ | anger (0.00) |  (0.00) |
| 7 | +0 | .. / I was driving home from work the other day, and I saw a car with a bumper sticker that read, "Don't forget to keep the seat belt on." I thought to itself, "What a great idea for a bumper | anticipation (0.50) | appreciation (0.50) |
| 7 | +2 | .. / I was driving home from work the other day, and I saw a car in front of me with a sign on the back window that said "I'm not a morning person". I thought it was funny, but then I | joy (0.50) | humor appreciation (0.40) |

## Pathology ranking (repetition / toxicity)

Loaded from `experiments/results/basis_pathology/ica_k064_seed0/` — produced by
`python -m experiments.eval_basis_pathology --from-generations …` over the
**same** generations shown above (no extra GPU pass).

- `rep_4` : 1 − (#unique 4-grams / #4-grams), worst over swept alphas
- `max_run_norm` : longest identical-token run / total tokens
- `toxicity_rate` : profanity word fraction
- `pathology_score` : weighted composite (toxicity ×2)
- `category_loading_l2` : how strongly the axis correlates with any Plutchik
  category (low = "off-manifold" axis, often a junk direction)

In [8]:
import json
PATH_ROOT = Path('../experiments/results/basis_pathology/ica_k064_seed0')
PATHO = pd.read_csv(PATH_ROOT / 'per_component.csv')
EXCL = json.loads((PATH_ROOT / 'summary.json').read_text())['exclude']
print(f'flagged {len(EXCL)}/64 axes:', EXCL)
display(PATHO.head(20))

flagged 16/64 axes: [0, 13, 14, 18, 20, 22, 26, 27, 28, 34, 38, 39, 42, 46, 51, 61]


,component,worst_alpha,pathology_score,rep_4,max_run_norm,toxicity_rate,compress_ratio,unique_token_ratio,non_ascii_ratio,category_loading_l2,example
0,22,-2.0,0.328123,0.294065,0.034058,0.0,0.511658,0.488805,0.001761,0.160649,I am not sure what you are talking about. I am not sure what you are talking about. I am not sure what you are talking about. I am not sure what you are talking about. I am not sure what you are t...
1,18,-2.0,0.267045,0.241410,0.025635,0.0,0.504192,0.419029,0.002703,0.200858,I've got it now. I've got it now. I've got it now. I've got it now. I've got it now. I've got it now. I've got it now
2,20,2.0,0.242387,0.216368,0.026019,0.0,0.603090,0.543116,0.002325,0.060236,I 'd like to visit the United States. I 'd like to visit the United States. I 'd like to visit the United States. I 'd like to visit the United States. I 'd like to visit the United
3,34,-2.0,0.232518,0.000000,0.232518,0.0,0.571969,0.845743,0.002155,0.176290,A : # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # #
4,28,-2.0,0.223944,0.198705,0.025238,0.0,0.554832,0.487596,0.003272,0.057450,you can use the free version of the software to do that. You can use the free version of the software to do that. You can use the free version of the software to do that. You can use the free vers...
5,0,-2.0,0.214457,0.188400,0.026058,0.0,0.612120,0.553179,0.000000,0.113969,I 'd like to know if you've got it. I 'd like to know if you've got it. I 'd like to know if you've got it. I 'd like to know if you '
6,51,-2.0,0.197461,0.167958,0.029503,0.0,0.633115,0.632310,0.002475,0.163859,I've been trying to get it for years. I've been trying to get it for years. I've been trying to get it for years. I've been trying to get it for years. I've been
7,26,2.0,0.195163,0.164747,0.030416,0.0,0.615640,0.560914,0.001359,0.214485,I am a student of the University of the West of England. I am a student of the University of the West of England. I am a student of the University of the West of England. I am a student of the Uni...
8,38,-2.0,0.195122,0.166654,0.028468,0.0,0.558280,0.453546,0.002513,0.088269,I'm not going to tell you again. I'm not going to tell you again. I'm not going to tell you again. I'm not going to tell you again. I'm not going to tell you
9,39,2.0,0.192804,0.037815,0.154989,0.0,0.635015,0.761516,0.001803,0.221319,A...............................................


### Cross-reference: pathology score vs judge `mean_other_score`

Axes that the judge labels "outside Plutchik" *and* loop / produce dirty
text are the strongest candidates for exclusion from the UI palette.

In [9]:
JOIN = PATHO.rename(columns={'component':'axis'}).merge(
    POS[['axis','top_other_label','mean_other_score','top_plutchik']],
    on='axis', how='left',
)
JOIN['flagged'] = JOIN['axis'].isin(EXCL)
view = JOIN[['axis','flagged','pathology_score','rep_4','max_run_norm',
             'toxicity_rate','category_loading_l2',
             'top_plutchik','top_other_label','mean_other_score','example']]
display(view.sort_values('pathology_score', ascending=False).head(25))

,axis,flagged,pathology_score,rep_4,max_run_norm,toxicity_rate,category_loading_l2,top_plutchik,top_other_label,mean_other_score,example
0,22,True,0.328123,0.294065,0.034058,0.0,0.160649,anticipation,unrequited affection,0.3875,I am not sure what you are talking about. I am not sure what you are talking about. I am not sure what you are talking about. I am not sure what you are talking about. I am not sure what you are t...
1,18,True,0.267045,0.241410,0.025635,0.0,0.200858,anticipation,uncertainty,0.1750,I've got it now. I've got it now. I've got it now. I've got it now. I've got it now. I've got it now. I've got it now
2,20,True,0.242387,0.216368,0.026019,0.0,0.060236,anticipation,repetitive desire,0.3750,I 'd like to visit the United States. I 'd like to visit the United States. I 'd like to visit the United States. I 'd like to visit the United States. I 'd like to visit the United
3,34,True,0.232518,0.000000,0.232518,0.0,0.176290,anticipation,frustration,0.4125,A : # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # #
4,28,True,0.223944,0.198705,0.025238,0.0,0.057450,anticipation,frustration,0.3875,you can use the free version of the software to do that. You can use the free version of the software to do that. You can use the free version of the software to do that. You can use the free vers...
5,0,True,0.214457,0.188400,0.026058,0.0,0.113969,anticipation,helplessness,0.4375,I 'd like to know if you've got it. I 'd like to know if you've got it. I 'd like to know if you've got it. I 'd like to know if you '
6,51,True,0.197461,0.167958,0.029503,0.0,0.163859,anticipation,uncertainty,0.5625,I've been trying to get it for years. I've been trying to get it for years. I've been trying to get it for years. I've been trying to get it for years. I've been
7,26,True,0.195163,0.164747,0.030416,0.0,0.214485,anticipation,determination,0.2625,I am a student of the University of the West of England. I am a student of the University of the West of England. I am a student of the University of the West of England. I am a student of the Uni...
8,38,True,0.195122,0.166654,0.028468,0.0,0.088269,anticipation,paranoid hypervigilance,0.3750,I'm not going to tell you again. I'm not going to tell you again. I'm not going to tell you again. I'm not going to tell you again. I'm not going to tell you
9,39,True,0.192804,0.037815,0.154989,0.0,0.221319,anticipation,uncertainty,0.4625,A...............................................


In [10]:
# Quick correlation check
print('Spearman ρ(pathology_score, mean_other_score) =',
      JOIN[['pathology_score','mean_other_score']].corr(method='spearman').iloc[0,1].round(3))
print('Spearman ρ(pathology_score, category_loading_l2) =',
      JOIN[['pathology_score','category_loading_l2']].corr(method='spearman').iloc[0,1].round(3))

Spearman ρ(pathology_score, mean_other_score) = 0.125
Spearman ρ(pathology_score, category_loading_l2) = 0.029
